In [4]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForSeq2SeqLM, AutoModelForMaskedLM, AutoModelForCausalLM
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch
import os


os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# 1. 加载模型和分词器
model_name = "EleutherAI/gpt-neo-125M"
model_name = "facebook/opt-125m"
# model_name = "cerebras/Cerebras-GPT-111M"
# model_name = "bigscience/bloom-560m" # I should try it on CoLab
# model_name = "mosaicml/mpt-7b"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

# model_name = "t5-base"
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# model_name = "microsoft/deberta-base"
# model = AutoModelForMaskedLM.from_pretrained(model_name)


tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


# 2. 加载数据集
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 3. 分词
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 4. 创建 DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. 创建 DataLoader
tokenized_datasets.set_format("torch")
dataloader = DataLoader(tokenized_datasets, batch_size=10, shuffle=True, collate_fn=data_collator)

# 6. 设置优化器
optimizer = AdamW(model.parameters(), lr=5e-5)

# 7. 训练循环
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device = torch.device("cpu")
model.to(device)
model.train()

from experiments.trainer.plugins import SnapshotPlugin, ProfilerPlugin
from perf_estimator.config import Config
snap_conf = Config(save2tmp=False)
snapshot = SnapshotPlugin(config=snap_conf)
profiler = ProfilerPlugin(config=snap_conf)

epochs = 1
snapshot.start()
profiler.start()
for epoch in range(epochs):
    for index, batch in enumerate(dataloader):
        snapshot.step()
        profiler.step()
        with torch.set_grad_enabled(True):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            if index == 3:
                break
snapshot.stop()
profiler.stop()

print("Train Finsihed！")

Epoch 0, Loss: 4.175829887390137
Epoch 0, Loss: 4.083451747894287
Epoch 0, Loss: 4.079841136932373
Epoch 0, Loss: 4.161968231201172
Train Finsihed！


In [2]:
from transformers import AutoModelForCausalLM, AutoModel
from huggingface_hub import model_info
model_info("facebook/bart-base").pipeline_tag

'feature-extraction'

In [3]:
AutoModel.from_pretrained("openai-community/gpt2")

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)